In [1]:

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Find project root
current_path = Path.cwd()

PROJECT_ROOT = next(
    (
        path for path in [current_path, *current_path.parents]
        if (path / 'data' / 'raw').exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Make sure the notebook "
        "is inside Retail-Sales-Forecasting-ML."
    )

# Define paths
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'

# Create processed directory if it doesn't exist
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH)
print("Processed data:", PROCESSED_DATA_PATH)


Project root: D:\projets\Retail-Sales-Forecasting-ML
Raw data: D:\projets\Retail-Sales-Forecasting-ML\data\raw
Processed data: D:\projets\Retail-Sales-Forecasting-ML\data\processed


In [5]:
# Define project paths
from pathlib import Path
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_DIR = ROOT_DIR / "data" / "raw"

# Load datasets
train = pd.read_csv(DATA_DIR / "train.csv")
store = pd.read_csv(DATA_DIR / "store.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

# Display confirmation
print("Datasets loaded successfully!")

# Display shapes
print("Train shape:", train.shape)
print("Store shape:", store.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

# Display first rows
display(train.head())
display(store.head())
display(test.head())
display(sample_submission.head())


Datasets loaded successfully!
Train shape: (1017209, 9)
Store shape: (1115, 10)
Test shape: (41088, 8)


C:\Users\mrymh\AppData\Local\Temp\ipykernel_10356\1830295326.py:11: DtypeWarning: Columns (7: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(DATA_DIR / "train.csv")


NameError: name 'sample_submission' is not defined

In [6]:

print("TRAIN DATA")
print(train.info())

print("\nSTORE DATA")
print(store.info())

print("\nTEST DATA")
print(test.info())

print("\nTrain columns:")
print(train.columns.tolist())

print("\nStore columns:")
print(store.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())


TRAIN DATA
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  str   
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  object
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), object(1), str(1)
memory usage: 69.8+ MB
None

STORE DATA
<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str  

In [7]:

# Check completely duplicated rows
print("Exact duplicate rows:")

print("Train:", train.duplicated().sum())
print("Store:", store.duplicated().sum())
print("Test:", test.duplicated().sum())

# Check duplicate store-date combinations
print("\nDuplicate Store-Date combinations:")

print(
    "Train:",
    train.duplicated(subset=['Store', 'Date']).sum()
)

print(
    "Test:",
    test.duplicated(subset=['Store', 'Date']).sum()
)

# Check duplicate store IDs
print(
    "\nDuplicate Store IDs:",
    store['Store'].duplicated().sum()
)


Exact duplicate rows:
Train: 0
Store: 0
Test: 0

Duplicate Store-Date combinations:
Train: 0
Test: 0

Duplicate Store IDs: 0


In [8]:

assert not store['Store'].duplicated().any(), \
    "Duplicate Store IDs detected!"

assert not train.duplicated(subset=['Store', 'Date']).any(), \
    "Duplicate Store-Date combinations in train!"

assert not test.duplicated(subset=['Store', 'Date']).any(), \
    "Duplicate Store-Date combinations in test!"

print("Duplicate validation passed!")


Duplicate validation passed!


In [9]:

# Merge store information into training data
train_merged = train.merge(
    store,
    on='Store',
    how='left',
    validate='many_to_one',
    indicator=True
)

# Check whether all stores were matched
print("Train merge results:")
print(train_merged['_merge'].value_counts())

assert (train_merged['_merge'] == 'both').all(), \
    "Some training stores were not matched!"

train_merged.drop(columns=['_merge'], inplace=True)


# Merge store information into test data
test_merged = test.merge(
    store,
    on='Store',
    how='left',
    validate='many_to_one',
    indicator=True
)

print("\nTest merge results:")
print(test_merged['_merge'].value_counts())

assert (test_merged['_merge'] == 'both').all(), \
    "Some test stores were not matched!"

test_merged.drop(columns=['_merge'], inplace=True)


# Check final shapes
print("\nMerged training shape:", train_merged.shape)
print("Merged test shape:", test_merged.shape)


Train merge results:
_merge
both          1017209
left_only           0
right_only          0
Name: count, dtype: int64

Test merge results:
_merge
both          41088
left_only         0
right_only        0
Name: count, dtype: int64

Merged training shape: (1017209, 18)
Merged test shape: (41088, 17)


C:\Users\mrymh\AppData\Local\Temp\ipykernel_10356\3985478963.py:17: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  train_merged.drop(columns=['_merge'], inplace=True)
C:\Users\mrymh\AppData\Local\Temp\ipykernel_10356\3985478963.py:35: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  test_merged.drop(columns=['_merge'], inplace=True)


In [10]:

# Convert Date to datetime
train_merged['Date'] = pd.to_datetime(
    train_merged['Date'],
    errors='raise'
)

test_merged['Date'] = pd.to_datetime(
    test_merged['Date'],
    errors='raise'
)


# Normalize StateHoliday
# The original data contains both 0 and '0'.
# We want all values to be consistent strings.

for df in [train_merged, test_merged]:

    df['StateHoliday'] = (
        df['StateHoliday']
        .astype(str)
        .str.strip()
        .replace({'0.0': '0'})
    )


# Convert numerical columns
numeric_columns = [
    'Store',
    'DayOfWeek',
    'Sales',
    'Customers',
    'Promo',
    'SchoolHoliday',
    'CompetitionDistance',
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2',
    'Promo2SinceWeek',
    'Promo2SinceYear'
]

for df in [train_merged, test_merged]:

    for col in numeric_columns:

        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors='raise'
            )


# Convert categorical columns
categorical_columns = [
    'StoreType',
    'Assortment',
    'PromoInterval'
]

for df in [train_merged, test_merged]:

    for col in categorical_columns:

        if col in df.columns:
            df[col] = df[col].astype('string')


print("Data types standardized!")

print("\nTraining data:")
print(train_merged.dtypes)

print("\nTest data:")
print(test_merged.dtypes)


Data types standardized!

Training data:
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                            str
SchoolHoliday                         int64
StoreType                            string
Assortment                           string
CompetitionDistance                 float64
CompetitionOpenSinceMonth           float64
CompetitionOpenSinceYear            float64
Promo2                                int64
Promo2SinceWeek                     float64
Promo2SinceYear                     float64
PromoInterval                        string
dtype: object

Test data:
Id                                    int64
Store                                 int64
DayOfWeek                             int64
Date     

In [11]:

def missing_value_report(df, dataset_name):

    missing = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing Percentage': (
            df.isnull().sum() / len(df) * 100
        )
    })

    missing = missing[
        missing['Missing Count'] > 0
    ].sort_values(
        by='Missing Percentage',
        ascending=False
    )

    print(f"\nMissing Values in {dataset_name}")
    display(missing)

    return missing


train_missing = missing_value_report(
    train_merged,
    'Training Dataset'
)

test_missing = missing_value_report(
    test_merged,
    'Test Dataset'
)



Missing Values in Training Dataset


,Missing Count,Missing Percentage
Promo2SinceYear,508031,49.94
Promo2SinceWeek,508031,49.94
PromoInterval,508031,49.94
CompetitionOpenSinceMonth,323348,31.79
CompetitionOpenSinceYear,323348,31.79
CompetitionDistance,2642,0.26



Missing Values in Test Dataset


,Missing Count,Missing Percentage
PromoInterval,17232,41.94
Promo2SinceYear,17232,41.94
Promo2SinceWeek,17232,41.94
CompetitionOpenSinceMonth,15216,37.03
CompetitionOpenSinceYear,15216,37.03
CompetitionDistance,96,0.23
Open,11,0.03


In [27]:

# Track missing competition distance
for df in [train_merged, test_merged]:

    df['CompetitionDistanceWasMissing'] = (
        df['CompetitionDistance']
        .isna()
        .astype('int8')
    )


# Calculate median from the original store-level dataset
competition_distance_median = store[
    'CompetitionDistance'
].median()

print(
    "Competition Distance Median:",
    competition_distance_median
)


# Fill CompetitionDistance
for df in [train_merged, test_merged]:

    df['CompetitionDistance'] = (
        df['CompetitionDistance']
        .fillna(competition_distance_median)
    )


# Fill competition opening information
competition_columns = [
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear'
]

for df in [train_merged, test_merged]:

    for col in competition_columns:

        df[col] = df[col].fillna(0).astype('int64')


# Fill Promo2 information
promo2_numeric_columns = [
    'Promo2SinceWeek',
    'Promo2SinceYear'
]

# Fill missing PromoInterval values
for df in [train_merged, test_merged]:
    df['PromoInterval'] = (
        df['PromoInterval']
        .fillna('NoPromo2')
        .astype(str)
    )

for df in [train_merged, test_merged]:

    for col in promo2_numeric_columns:

        df[col] = df[col].fillna(0).astype('int64')


for df in [train_merged, test_merged]:

    df['PromoInterval'] = (
        df['PromoInterval']
        .fillna('NoPromo2')
        .astype(str)
    )

print("PromoInterval missing values handled!")

print("Competition and Promo2 missing values handled!")


Competition Distance Median: 2325.0
PromoInterval missing values handled!
Competition and Promo2 missing values handled!


In [21]:

# Build historical opening pattern from training data
open_lookup = (
    train_merged
    .dropna(subset=['Open'])
    .groupby(['Store', 'DayOfWeek'])['Open']
    .agg(lambda x: x.mode().iloc[0])
)

# Overall fallback
global_open_mode = int(
    train_merged['Open'].mode().iloc[0]
)

print("Global Open mode:", global_open_mode)


def fill_missing_open(df):

    df = df.copy()

    # Record which values were originally missing
    missing_mask = df['Open'].isna()

    df['OpenWasMissing'] = (
        missing_mask.astype('int8')
    )

    if missing_mask.any():

        # Find historical opening status for each Store-DayOfWeek
        keys = pd.MultiIndex.from_frame(
            df.loc[missing_mask, ['Store', 'DayOfWeek']]
        )

        mapped_values = open_lookup.reindex(keys)

        # Use historical values where available
        # Otherwise use the global mode
        mapped_values = mapped_values.fillna(
            global_open_mode
        )

        df.loc[missing_mask, 'Open'] = (
            mapped_values.to_numpy()
        )

    df['Open'] = df['Open'].astype('int8')

    return df


# Apply to both datasets
train_merged = fill_missing_open(train_merged)
test_merged = fill_missing_open(test_merged)


print("Remaining missing Open values:")
print("Train:", train_merged['Open'].isna().sum())
print("Test:", test_merged['Open'].isna().sum())


Global Open mode: 1
Remaining missing Open values:
Train: 0
Test: 0


In [14]:

print("TRAIN - Remaining Missing Values")
print(
    train_merged.isnull().sum()[
        train_merged.isnull().sum() > 0
    ]
)

print("\nTEST - Remaining Missing Values")
print(
    test_merged.isnull().sum()[
        test_merged.isnull().sum() > 0
    ]
)


TRAIN - Remaining Missing Values
Series([], dtype: int64)

TEST - Remaining Missing Values
Series([], dtype: int64)


In [22]:

# Check binary features
binary_columns = [
    'Open',
    'Promo',
    'SchoolHoliday',
    'Promo2'
]

for df_name, df in [
    ('Train', train_merged),
    ('Test', test_merged)
]:

    print(f"\n{df_name} binary feature validation:")

    for col in binary_columns:

        print(
            col,
            sorted(df[col].dropna().unique())
        )


# Check DayOfWeek
print("\nDayOfWeek values:")
print(sorted(train_merged['DayOfWeek'].unique()))

# Check StateHoliday
print("\nStateHoliday values:")
print(train_merged['StateHoliday'].unique())

# Check StoreType
print("\nStoreType values:")
print(train_merged['StoreType'].unique())

# Check Assortment
print("\nAssortment values:")
print(train_merged['Assortment'].unique())



Train binary feature validation:
Open [np.int8(0), np.int8(1)]
Promo [np.int64(0), np.int64(1)]
SchoolHoliday [np.int64(0), np.int64(1)]
Promo2 [np.int64(0), np.int64(1)]

Test binary feature validation:
Open [np.int8(0), np.int8(1)]
Promo [np.int64(0), np.int64(1)]
SchoolHoliday [np.int64(0), np.int64(1)]
Promo2 [np.int64(0), np.int64(1)]

DayOfWeek values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

StateHoliday values:
<StringArray>
['0', 'a', 'b', 'c']
Length: 4, dtype: str

StoreType values:
<StringArray>
['c', 'a', 'd', 'b']
Length: 4, dtype: string

Assortment values:
<StringArray>
['a', 'c', 'b']
Length: 3, dtype: string


In [23]:

for df in [train_merged, test_merged]:

    for col in binary_columns:

        assert df[col].isin([0, 1]).all(), \
            f"Invalid values found in {col}!"

    assert df['DayOfWeek'].between(1, 7).all(), \
        "Invalid DayOfWeek values!"

    assert df['StateHoliday'].isin(
        ['0', 'a', 'b', 'c']
    ).all(), "Unexpected StateHoliday values!"

    assert df['StoreType'].isin(
        ['a', 'b', 'c', 'd']
    ).all(), "Unexpected StoreType values!"

    assert df['Assortment'].isin(
        ['a', 'b', 'c']
    ).all(), "Unexpected Assortment values!"


# Training-specific validation
assert (train_merged['Sales'] >= 0).all()
assert (train_merged['Customers'] >= 0).all()

print("All data validation checks passed!")


All data validation checks passed!


In [24]:

print("=" * 50)
print("FINAL PREPROCESSING SUMMARY")
print("=" * 50)

print("\nTraining dataset:")
print("Shape:", train_merged.shape)
print("Missing values:", train_merged.isnull().sum().sum())

print("\nTest dataset:")
print("Shape:", test_merged.shape)
print("Missing values:", test_merged.isnull().sum().sum())

print("\nTraining columns:")
print(train_merged.columns.tolist())

print("\nTest columns:")
print(test_merged.columns.tolist())

print("\nTraining sample:")
display(train_merged.head())

print("\nTest sample:")
display(test_merged.head())


FINAL PREPROCESSING SUMMARY

Training dataset:
Shape: (1017209, 20)
Missing values: 0

Test dataset:
Shape: (41088, 19)
Missing values: 0

Training columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionDistanceWasMissing', 'OpenWasMissing']

Test columns:
['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionDistanceWasMissing', 'OpenWasMissing']

Training sample:


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceWasMissing,OpenWasMissing
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.00,9,2008,0,0,0,None,0,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.00,11,2007,1,13,2010,"Jan,Apr,Jul,Oct",0,0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.00,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",0,0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.00,9,2009,0,0,0,None,0,0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.00,4,2015,0,0,0,None,0,0



Test sample:


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceWasMissing,OpenWasMissing
0,1,1,4,2015-09-17,1,1,0,0,c,a,1270.00,9,2008,0,0,0,None,0,0
1,2,3,4,2015-09-17,1,1,0,0,a,a,14130.00,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",0,0
2,3,7,4,2015-09-17,1,1,0,0,a,c,24000.00,4,2013,0,0,0,None,0,0
3,4,8,4,2015-09-17,1,1,0,0,a,a,7520.00,10,2014,0,0,0,None,0,0
4,5,9,4,2015-09-17,1,1,0,0,a,c,2030.00,8,2000,0,0,0,None,0,0


In [25]:

# Output file paths
train_output = (
    PROCESSED_DATA_PATH / 'train_preprocessed.csv.gz'
)

test_output = (
    PROCESSED_DATA_PATH / 'test_preprocessed.csv.gz'
)

# Save datasets
train_merged.to_csv(
    train_output,
    index=False,
    compression='gzip'
)

test_merged.to_csv(
    test_output,
    index=False,
    compression='gzip'
)

print("Processed datasets saved successfully!")

print("\nSaved files:")
print(train_output)
print(test_output)


Processed datasets saved successfully!

Saved files:
D:\projets\Retail-Sales-Forecasting-ML\data\processed\train_preprocessed.csv.gz
D:\projets\Retail-Sales-Forecasting-ML\data\processed\test_preprocessed.csv.gz


In [28]:

# Reload processed datasets

train_check = pd.read_csv(
    train_output,
    parse_dates=['Date'],
    dtype={'StateHoliday': 'string'}
)

test_check = pd.read_csv(
    test_output,
    parse_dates=['Date'],
    dtype={'StateHoliday': 'string'}
)

# Verify shapes
assert train_check.shape == train_merged.shape
assert test_check.shape == test_merged.shape

# Verify target column
assert 'Sales' in train_check.columns
assert 'Sales' not in test_check.columns

# Check missing values
print("Missing values in reloaded training data:")
print(train_check.isnull().sum()[train_check.isnull().sum() > 0])

print("\nMissing values in reloaded test data:")
print(test_check.isnull().sum()[test_check.isnull().sum() > 0])

# Verify no missing values
assert train_check.isnull().sum().sum() == 0
assert test_check.isnull().sum().sum() == 0

print("\nFinal verification passed!")

print("Training shape:", train_check.shape)
print("Test shape:", test_check.shape)



Missing values in reloaded training data:
PromoInterval    508031
dtype: int64

Missing values in reloaded test data:
PromoInterval    17232
dtype: int64


AssertionError: 